## Section 1: The Configuration & Environment Setup
*We centralize all hyperparameters here. This makes Ablation Studies easy. We also initialize our GPU and import required libraries.*

In [16]:
import os
import torch
import numpy as np
import random
import wandb
from google.colab import drive
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- THE CONTROL PANEL ---
CONFIG = {
    "epochs": 20,
    "batch_size": 128,            # REDUCED: DGCNN uses way more VRAM!
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "n_points": 1024,
    "num_classes": 40,
    "k": 20,                     # NEW: Number of nearest neighbors
    "use_weighted_loss": True,
    "use_augmentation": True,
    "run_name": "dgcnn"
}

# Setup GPU
# Dynamically checks for Apple Silicon (mps), then NVIDIA (cuda), then CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


## Section 2: Data Extraction
*Mount Google Drive and extract the ZIP file to the fast local Colab storage (`/content/data`) to prevent I/O bottlenecks.*

In [17]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
#zip_path = '/content/drive/MyDrive/Colab Notebooks/AI535_Final_Project/ModelNet40.zip'
zip_path = '/content/drive/MyDrive/ModelNet40.zip'
local_extract_path = '/content/data'
root_dir = '/content/data/ModelNet40'

# 3. Extract if not already extracted in this session
if not os.path.exists(root_dir):
    print("Extracting ModelNet40 to local Colab storage. This may take a minute...")
    !unzip -q "{zip_path}" -d {local_extract_path}
    print("Extraction complete!")
else:
    print("Data is already extracted and ready to go.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data is already extracted and ready to go.


## Section 3: High-RAM Dataset Loading & Class Weight Calculation
*Parse the 3D objects, cache them in RAM, and mathematically calculate the inverse class frequencies to handle ModelNet40's severe class imbalance.*

In [18]:
# Dynamically map the 40 classes
classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

def load_off(file_path, n_points=1024):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    if lines[0].strip() == 'OFF':
        n_verts = int(lines[1].strip().split()[0])
        verts_start = 2
    else:
        n_verts = int(lines[0].strip()[3:].split()[0])
        verts_start = 1

    verts = np.array([ [float(x) for x in l.strip().split()] for l in lines[verts_start:verts_start + n_verts] ])

    if len(verts) >= n_points:
        indices = np.random.choice(len(verts), n_points, replace=False)
    else:
        indices = np.random.choice(len(verts), n_points, replace=True)
    verts = verts[indices]

    # Normalization
    verts = verts - np.mean(verts, axis=0)
    dist = np.max(np.sqrt(np.sum(verts**2, axis=1)))
    verts = verts / dist

    return torch.tensor(verts, dtype=torch.float32)

class FastModelNetDataset(Dataset):
    def __init__(self, root, split='train', n_points=1024):
        self.n_points = n_points
        self.split = split
        self.files = []
        for cls_name, idx in class_to_idx.items():
            cls_folder = os.path.join(root, cls_name, split)
            if os.path.exists(cls_folder):
                for f in os.listdir(cls_folder):
                    if f.endswith('.off'):
                        self.files.append({'path': os.path.join(cls_folder, f), 'label': idx})

        print(f"Parsing and caching {len(self.files)} {split} files into RAM...")
        self.data = []
        self.labels = []
        for item in self.files:
            points = load_off(item['path'], self.n_points)
            self.data.append(points.transpose(0, 1))
            self.labels.append(item['label'])
        print(f"Finished caching {split} split!")

    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        # Clone the data so we don't accidentally overwrite our clean RAM cache!
        points = self.data[idx].clone()
        label = self.labels[idx]

        # --- DYNAMIC DATA AUGMENTATION (Training Set Only) ---
        if self.split == 'train' and CONFIG.get("use_augmentation", False):
            # 1. Random Rotation (around the Up/Y-axis)
            theta = random.uniform(0, 2 * np.pi)
            rot_matrix = torch.tensor([
                [np.cos(theta), 0, np.sin(theta)],
                [0, 1, 0],
                [-np.sin(theta), 0, np.cos(theta)]
            ], dtype=torch.float32)

            # Multiply (3,3) matrix by our (3, 1024) point cloud
            points = torch.matmul(rot_matrix, points)

            # 2. Random Gaussian Jitter (Noise)
            noise = torch.randn_like(points) * 0.02
            points = points + noise

        return points, label

# Initialize Datasets
trainset = FastModelNetDataset(root_dir, split='train', n_points=CONFIG["n_points"])
testset = FastModelNetDataset(root_dir, split='test', n_points=CONFIG["n_points"])

trainloader = DataLoader(trainset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)

# --- CLASS IMBALANCE CALCULATION ---
print("\nCalculating class weights for Weighted Cross-Entropy...")
class_counts = np.zeros(CONFIG["num_classes"])
for label in trainset.labels:
    class_counts[label] += 1

# w_c = N_total / (C * N_c)
total_samples = len(trainset.labels)
class_weights = total_samples / (CONFIG["num_classes"] * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights calculated successfully!")

Parsing and caching 1095 train files into RAM...


KeyboardInterrupt: 

## Section 4: PointNet Architecture
*Defines the Spatial Deep Network using Shared MLPs and Global Max Pooling to achieve permutation invariance.*

In [ ]:
def knn(x, k):
    # Mathematically computes the pairwise distance between all points
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)

    # Returns the indices of the k closest points
    idx = pairwise_distance.topk(k=k, dim=-1)[1]   # (batch_size, num_points, k)
    return idx

def get_graph_feature(x, k=20, idx=None):
    batch_size = x.size(0)
    num_points = x.size(2)
    x = x.view(batch_size, -1, num_points)

    if idx is None:
        idx = knn(x, k=k)   # Get nearest neighbors

    device = x.device
    idx_base = torch.arange(0, batch_size, device=device).view(-1, 1, 1) * num_points
    idx = idx + idx_base
    idx = idx.view(-1)

    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(batch_size * num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims)
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1)

    # Concatenate local edge features (neighbor - center) with global center features
    feature = torch.cat((feature - x, x), dim=3).permute(0, 3, 1, 2).contiguous()
    return feature

class DGCNN(nn.Module):
    def __init__(self, num_classes=40, k=20):
        super(DGCNN, self).__init__()
        self.k = k

        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        self.bn5 = nn.BatchNorm1d(1024)

        # EdgeConv Layers (Note they use Conv2d because of the k-dimension)
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, kernel_size=1, bias=False), self.bn1, nn.LeakyReLU(negative_slope=0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64*2, 64, kernel_size=1, bias=False), self.bn2, nn.LeakyReLU(negative_slope=0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(64*2, 128, kernel_size=1, bias=False), self.bn3, nn.LeakyReLU(negative_slope=0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(128*2, 256, kernel_size=1, bias=False), self.bn4, nn.LeakyReLU(negative_slope=0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, 1024, kernel_size=1, bias=False), self.bn5, nn.LeakyReLU(negative_slope=0.2))

        # Classification Head
        self.linear1 = nn.Linear(1024*2, 512, bias=False)
        self.bn6 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(p=0.5)
        self.linear2 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(p=0.5)
        self.linear3 = nn.Linear(256, num_classes)

    def forward(self, x):
        batch_size = x.size(0)

        # 1st EdgeConv
        x = get_graph_feature(x, k=self.k)
        x = self.conv1(x)
        x1 = x.max(dim=-1, keepdim=False)[0]

        # 2nd EdgeConv
        x = get_graph_feature(x1, k=self.k)
        x = self.conv2(x)
        x2 = x.max(dim=-1, keepdim=False)[0]

        # 3rd EdgeConv
        x = get_graph_feature(x2, k=self.k)
        x = self.conv3(x)
        x3 = x.max(dim=-1, keepdim=False)[0]

        # 4th EdgeConv
        x = get_graph_feature(x3, k=self.k)
        x = self.conv4(x)
        x4 = x.max(dim=-1, keepdim=False)[0]

        # Aggregate all EdgeConv layers
        x = torch.cat((x1, x2, x3, x4), dim=1)
        x = self.conv5(x)

        # DGCNN uses both Max Pooling AND Avg Pooling for better features
        x1 = F.adaptive_max_pool1d(x, 1).view(batch_size, -1)
        x2 = F.adaptive_avg_pool1d(x, 1).view(batch_size, -1)
        x = torch.cat((x1, x2), 1)

        # Pass through MLPs
        x = F.leaky_relu(self.bn6(self.linear1(x)), negative_slope=0.2)
        x = self.drop1(x)
        x = F.leaky_relu(self.bn7(self.linear2(x)), negative_slope=0.2)
        x = self.drop2(x)
        out = self.linear3(x)

        return out

model = DGCNN(num_classes=CONFIG["num_classes"], k=CONFIG["k"]).to(device)
print("DGCNN Initialized!")

DGCNN Initialized!


## Section 5: Training Loop with Early Stopping
*Trains the model using AdamW and Cosine Annealing. Implements checkpointing to save the best model weights before overfitting occurs.*

In [ ]:
wandb.login()
wandb.init(project="ai535-final-project", name=CONFIG["run_name"], config=CONFIG)

# --- APPLY WEIGHTED LOSS IF CONFIGURED ---
if CONFIG["use_weighted_loss"]:
    print("Using WEIGHTED Cross-Entropy Loss with Label Smoothing")
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
else:
    print("Using STANDARD Cross-Entropy Loss")
    criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

wandb.watch(model, criterion, log="all", log_freq=10)

best_test_error = 100.0
#save_path = '/content/drive/MyDrive/Colab Notebooks/AI535_Final_Project/best_dgcnn_model.pth'
save_path = '/content/drive/MyDrive/best_dgcnn_model.pth'

print(f"Start Training for {CONFIG['epochs']} Epochs...")

for epoch in range(CONFIG["epochs"]):
    model.train()
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    avg_train_loss = running_loss / len(trainset)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # Evaluation phase
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            test_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_test_loss = test_loss / len(testset)
    test_error = 100.0 * (1 - correct / total)

    wandb.log({
        "Epoch": epoch + 1,
        "Training Loss": avg_train_loss,
        "Testing Loss": avg_test_loss,
        "Testing Error (%)": test_error,
        "Learning Rate": current_lr
    })

    # --- EARLY STOPPING CHECKPOINTING ---
    if test_error < best_test_error:
        best_test_error = test_error
        torch.save(model.state_dict(), save_path)
        checkpoint_msg = f"** New Best! Saved to Drive **"
    else:
        checkpoint_msg = ""

    print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] - Train Loss: {avg_train_loss:.4f} | Test Error: {test_error:.2f}% | {checkpoint_msg}")

print(f"Finished Training! Best Test Error was {best_test_error:.2f}%")
wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 ㄉ


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hsuyo (hsuyo-oregon-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using WEIGHTED Cross-Entropy Loss with Label Smoothing
Start Training for 200 Epochs...
Epoch [1/200] - Train Loss: nan | Test Error: 100.00% | 
Epoch [2/200] - Train Loss: nan | Test Error: 100.00% | 
Epoch [3/200] - Train Loss: nan | Test Error: 100.00% | 
Epoch [4/200] - Train Loss: nan | Test Error: 100.00% | 


KeyboardInterrupt: 

## Section 6: Quantitative Evaluation (OA & mAcc)
*Loads the best saved weights and calculates Overall Accuracy and Mean Per-Class Accuracy to prove the effectiveness of the Weighted Loss.*

In [ ]:
# 1. Surgically remove W&B hooks
for module in model.modules():
    module._forward_hooks.clear()
    module._backward_hooks.clear()
    module._forward_pre_hooks.clear()

# 2. Load the BEST weights from our checkpoint
model.load_state_dict(torch.load(save_path))
model.eval()

total, correct = 0, 0
class_correct = torch.zeros(40).to(device)
class_total = torch.zeros(40).to(device)

print("Evaluating Best Checkpoint on Test Set...")

with torch.no_grad():
    for points, labels in testloader:
        points, labels = points.to(device), labels.to(device)
        outputs = model(points)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        c = (predicted == labels)
        for i in range(labels.size(0)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

oa = 100.0 * correct / total
per_class_acc = class_correct / torch.clamp(class_total, min=1)
macc = 100.0 * torch.mean(per_class_acc).item()

print("-" * 30)
print(f"Final Overall Accuracy (OA):       {oa:.2f}%")
print(f"Final Mean Per-Class Acc (mAcc):   {macc:.2f}%")
print("-" * 30)

print("\nLowest performing classes:")
acc_dict = {classes[i]: (per_class_acc[i].item() * 100) for i in range(40)}
sorted_acc = sorted(acc_dict.items(), key=lambda x: x[1])
for cls_name, acc in sorted_acc[:5]:
    print(f"{cls_name:>15}: {acc:.2f}%")

Evaluating Best Checkpoint on Test Set...
------------------------------
Final Overall Accuracy (OA):       65.56%
Final Mean Per-Class Acc (mAcc):   63.18%
------------------------------

Lowest performing classes:
          table: 11.00%
     flower_pot: 15.00%
          radio: 25.00%
          stool: 25.00%
       tv_stand: 25.00%


## Section 7: Qualitative Results (10-Class Visualization Grid)
*Generates a 3D scatter plot grid of 10 distinct classes to visually inspect the model's performance.*

In [ ]:
all_candidates = []
print("Gathering predictions for visualization...")
with torch.no_grad():
  for points, labels in testloader:
    points_gpu = points.to(device)
    outputs = model(points_gpu) _, preds = torch.max(outputs, 1)
    for i in range(points.size(0)):
      all_candidates.append({ 'pc': points[i].cpu().numpy(), 'pred': preds[i].item(), 'true': labels[i].item() })
random.shuffle(all_candidates)
selected_samples = []
seen_preds = set()
      for cand in all_candidates:
        if cand['pred'] not in seen_preds:
          seen_preds.add(cand['pred'])
          selected_samples.append(cand)
        if len(selected_samples) == 10:
          break
fig = plt.figure(figsize=(20, 8))
fig.suptitle("DGCNN Predictions (10 Distinct Classes)", fontsize=20, y=1.05)
for i, sample in enumerate(selected_samples):
  ax = fig.add_subplot(2, 5, i+1, projection='3d')
  pc = sample['pc'].transpose(1, 0)
  ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=5, c=pc[:, 2], cmap='viridis')
  pred_name = classes[sample['pred']]
  true_name = classes[sample['true']]
  title_color = 'green' if sample['pred'] == sample['true'] else 'red'
  ax.set_title(f"Pred: {pred_name}\n(True: {true_name})",
color=title_color, fontsize=14, fontweight='bold')
  ax.axis('off')
plt.tight_layout()
plt.show()
fig.savefig("10_class_predictions_dgcnn.png", bbox_inches='tight', dpi=300)
print("Visualization saved as '10_class_predictions_dgcnn.png'")
all_candidates = []

print("Gathering predictions for visualization...")
with torch.no_grad():
    for points, labels in testloader:
        points_gpu = points.to(device)
        outputs = model(points_gpu)
        _, preds = torch.max(outputs, 1)

        for i in range(points.size(0)):
            all_candidates.append({
                'pc': points[i].cpu().numpy(),   # shape: [3, N] or [N, 3]
                'pred': preds[i].item(),
                'true': labels[i].item()
            })

# 先把正確的放前面，這樣比較容易挑到好看的例子
all_candidates.sort(key=lambda x: x['pred'] != x['true'])

selected_samples = []
seen_preds = set()

for cand in all_candidates:
    if cand['pred'] not in seen_preds:
        seen_preds.add(cand['pred'])
        selected_samples.append(cand)
    if len(selected_samples) == 10:
        break

fig = plt.figure(figsize=(22, 9))
fig.suptitle("DGCNN Predictions (10 Distinct Predicted Classes)", fontsize=20, y=0.98)

for i, sample in enumerate(selected_samples):
    ax = fig.add_subplot(2, 5, i + 1, projection='3d')

    pc = sample['pc']

    # 確保 shape 是 [N, 3]
    if pc.shape[0] == 3:
        pc = pc.transpose(1, 0)

    # Normalize point cloud
    pc = pc - np.mean(pc, axis=0)
    scale = np.max(np.linalg.norm(pc, axis=1))
    if scale > 0:
        pc = pc / scale

    x, y, z = pc[:, 0], pc[:, 1], pc[:, 2]

    # 用 z 當顏色，保留立體感
    ax.scatter(x, y, z, s=8, c=z, cmap='viridis', alpha=0.9)

    pred_name = classes[sample['pred']]
    true_name = classes[sample['true']]
    correct = sample['pred'] == sample['true']

    title_color = 'green' if correct else 'red'
    ax.set_title(
        f"Pred: {pred_name}\n(True: {true_name})",
        color=title_color,
        fontsize=13,
        fontweight='bold',
        pad=10
    )

    # 固定視角
    ax.view_init(elev=20, azim=45)

    # 固定範圍，讓不同樣本比較一致
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])

    # 移除刻度
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    # 移除背景 pane
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    # 移除格線
    ax.grid(False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("10_class_predictions_dgcnn_clear.png", bbox_inches='tight', dpi=300)
plt.show()

print("Visualization saved as '10_class_predictions_dgcnn_clear.png'")

Gathering predictions for visualization...


NameError: name 'testloader' is not defined